# Speed Limit Signs -- YOLOv5n Training

Train YOLOv5n for the AI-Powered HUD project.

**Data source:** [MTSD](https://www.mapillary.com/dataset/trafficsign) (Mapillary Traffic Sign Dataset)

**Target device:** Luckfox Pico Ultra (RV1106G3, 0.5 TOPS NPU, INT8)

**Model:** 11-class universal, covers AU + CN speed limits

**Training resolution:** 640x640

**Runtime:** Change to GPU (Runtime > Change runtime type > T4 GPU)

| ID | Class | AU | CN |
|----|-------|:--:|:--:|
| 0 | speed_sign_20 | - | Y |
| 1 | speed_sign_30 | Y | Y |
| 2 | speed_sign_40 | Y | Y |
| 3 | speed_sign_50 | Y | Y |
| 4 | speed_sign_60 | Y | Y |
| 5 | speed_sign_70 | Y | Y |
| 6 | speed_sign_80 | Y | Y |
| 7 | speed_sign_90 | Y | - |
| 8 | speed_sign_100 | Y | Y |
| 9 | speed_sign_110 | Y | Y |
| 10 | speed_sign_120 | - | Y |

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
TARGET_CLASSES = [
    "speed_sign_20",  "speed_sign_30",  "speed_sign_40",
    "speed_sign_50",  "speed_sign_60",  "speed_sign_70",
    "speed_sign_80",  "speed_sign_90",  "speed_sign_100",
    "speed_sign_110", "speed_sign_120",
]
UPLOAD_FILE = "speed_signs_dataset.tar.gz"
DATASET_DIR = "speed_signs_dataset"
PROJECT_NAME = "speed_signs"
RUN_NAME = "universal"  # YOLOv5 run directory name

NC = len(TARGET_CLASSES)
DATASET_ROOT = f"/content/{DATASET_DIR}"

# ============================================================
# TRAINING PARAMETERS
# ============================================================
EPOCHS = 300
BATCH_SIZE = 16
IMG_SIZE = 640
WORKERS = 2

# ============================================================
# GOOGLE DRIVE CHECKPOINT BACKUP
# Survives runtime recycling. On disconnect:
#   1. Re-run all cells from top (env + dataset auto-restore)
#   2. Training auto-resumes from last Drive checkpoint
# ============================================================
DRIVE_BACKUP_DIR = f"/content/drive/MyDrive/ai-hud-training/{PROJECT_NAME}_{RUN_NAME}"
BACKUP_INTERVAL_MIN = 5  # Sync last.pt to Drive every N minutes

print(f"Model:     {PROJECT_NAME} (11-class universal, AU + CN)")
print(f"Classes:   {NC}")
print(f"Training:  {EPOCHS} epochs, batch {BATCH_SIZE}, img {IMG_SIZE}")
print(f"Upload:    {UPLOAD_FILE}")
print(f"Dataset:   {DATASET_ROOT}")
print(f"Backup:    {DRIVE_BACKUP_DIR} (every {BACKUP_INTERVAL_MIN} min)")
print()
for i, name in enumerate(TARGET_CLASSES):
    print(f"  {i}: {name}")

## 1. Environment Setup

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f"VRAM: {vram / 1024**3:.1f} GB")

In [ ]:
import os

# Clone airockchip/yolov5 (RKNN-optimized fork, required for --rknpu export)
# Idempotent: skip if already cloned (avoids error on re-run after disconnect)
if os.path.exists("/content/yolov5/.git"):
    print("yolov5 repo already exists, skipping clone.")
else:
    !git clone https://github.com/airockchip/yolov5.git /content/yolov5

%cd /content/yolov5
!pip install -r requirements.txt -q

# [Fix] onnxscript is required by ONNX export with PyTorch >= 2.6
!pip install onnxscript -q

# [Fix] Pillow 10+ removed font.getsize() used by YOLOv5 utils/plots.py
!pip install "Pillow<10" -q

## 2. Load Dataset

**Option A (recommended):** Load from Google Drive (fast, reusable across sessions)

1. Upload `speed_signs_dataset.tar.gz` to your Google Drive root **once**
2. Run the cell below -- it mounts Drive and extracts automatically

**Option B:** Direct upload via browser (slow, ~3GB each time)

Set `USE_DRIVE = False` in the next cell to use browser upload instead.

In [ ]:
import os, yaml, shutil
from pathlib import Path
from collections import defaultdict

# ============================================================
# Toggle: True = Google Drive, False = browser upload
# ============================================================
USE_DRIVE = True
DRIVE_PATH = f"/content/drive/MyDrive/{UPLOAD_FILE}"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    if os.path.exists(DRIVE_PATH):
        print(f"Found {DRIVE_PATH}, extracting...")
        !tar -xzf "{DRIVE_PATH}" -C /content/
    else:
        raise FileNotFoundError(
            f"{DRIVE_PATH} not found.\n"
            f"Upload {UPLOAD_FILE} to your Google Drive root first,\n"
            f"or set USE_DRIVE = False to use browser upload.")
else:
    from google.colab import files
    print(f"Upload {UPLOAD_FILE} ...")
    uploaded = files.upload()
    if uploaded:
        fname = list(uploaded.keys())[0]
        !tar -xzf "{fname}" -C /content/
        !rm -f "{fname}"
    else:
        raise RuntimeError("No file uploaded.")

# Auto-detect extracted directory and rename to expected DATASET_DIR
# Handles tar.gz archives with any top-level directory name
if not os.path.exists(DATASET_ROOT):
    extracted = [
        d for d in os.listdir("/content/")
        if os.path.isdir(f"/content/{d}")
        and os.path.exists(f"/content/{d}/data.yaml")
        and d != DATASET_DIR
    ]
    if len(extracted) == 1:
        src = f"/content/{extracted[0]}"
        print(f"Renaming extracted directory: {extracted[0]} -> {DATASET_DIR}")
        os.rename(src, DATASET_ROOT)
    elif len(extracted) > 1:
        raise RuntimeError(
            f"Multiple dataset directories found: {extracted}. "
            f"Remove stale ones and retry.")
    else:
        raise FileNotFoundError(
            f"{DATASET_ROOT} not found after extraction, and no directory "
            f"with data.yaml detected under /content/.")

# Verify
for split in ["train", "val"]:
    img_dir = f"{DATASET_ROOT}/{split}/images"
    n = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
    print(f"  {split}: {n} images")

# Update paths in data.yaml for Colab absolute paths
data_yaml = {
    "train": f"{DATASET_ROOT}/train/images",
    "val": f"{DATASET_ROOT}/val/images",
    "nc": NC,
    "names": TARGET_CLASSES,
}
with open(f"{DATASET_ROOT}/data.yaml", "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)

print(f"\nDataset ready at {DATASET_ROOT}")

## 3. Dataset Statistics & Visualization

In [ ]:
# Class distribution
total_stats = defaultdict(int)

for split in ["train", "val"]:
    lbl_dir = f"{DATASET_ROOT}/{split}/labels"
    img_dir = f"{DATASET_ROOT}/{split}/images"
    if not os.path.exists(lbl_dir):
        continue

    n_images = len([f for f in os.listdir(img_dir)
                    if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    split_counts = defaultdict(int)

    for lbl_file in os.listdir(lbl_dir):
        if not lbl_file.endswith(".txt"):
            continue
        try:
            with open(os.path.join(lbl_dir, lbl_file), "r", encoding="utf-8", errors="ignore") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cls_id = int(parts[0])
                        split_counts[cls_id] += 1
                        total_stats[cls_id] += 1
        except (UnicodeDecodeError, ValueError):
            continue

    print(f"\n  {split}: {n_images} images")
    for cls_id in range(NC):
        print(f"    {cls_id}: {TARGET_CLASSES[cls_id]:<18s} {split_counts.get(cls_id, 0):>5d}")

print(f"\n{'='*55}")
total_ann = sum(total_stats.values())
for cls_id in range(NC):
    name = TARGET_CLASSES[cls_id]
    count = total_stats.get(cls_id, 0)
    pct = (count / total_ann * 100) if total_ann > 0 else 0
    bar = '#' * min(count // 10, 40)
    print(f"  {cls_id}: {name:<18s} {count:>5d} ({pct:>5.1f}%) {bar}")
print(f"  {'TOTAL':<21s} {total_ann:>5d}")

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import random

COLORS = [
    (255, 80, 80), (80, 255, 80), (80, 80, 255), (255, 255, 80),
    (255, 80, 255), (80, 255, 255), (255, 160, 80), (160, 80, 255),
    (80, 160, 255), (200, 200, 80), (200, 80, 200),
]

def draw_yolo_boxes(img_path, label_path, class_names):
    img = cv2.imread(img_path)
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    if os.path.exists(label_path):
        with open(label_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls_id = int(parts[0])
                cx, cy, bw, bh = [float(x) for x in parts[1:5]]
                x1 = int((cx - bw / 2) * w)
                y1 = int((cy - bh / 2) * h)
                x2 = int((cx + bw / 2) * w)
                y2 = int((cy + bh / 2) * h)
                color = COLORS[cls_id % len(COLORS)]
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                label = class_names[cls_id] if cls_id < len(class_names) else f"cls{cls_id}"
                cv2.putText(img, label, (x1, y1 - 5),
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    return img

train_img_dir = f"{DATASET_ROOT}/train/images"
train_lbl_dir = f"{DATASET_ROOT}/train/labels"
all_imgs = [f for f in os.listdir(train_img_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

if all_imgs:
    samples = random.sample(all_imgs, min(8, len(all_imgs)))
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    for ax, img_file in zip(axes.flat, samples):
        stem = Path(img_file).stem
        img = draw_yolo_boxes(
            os.path.join(train_img_dir, img_file),
            os.path.join(train_lbl_dir, stem + ".txt"),
            TARGET_CLASSES,
        )
        if img is not None:
            ax.imshow(img)
            ax.set_title(img_file[:30], fontsize=8)
        ax.axis("off")
    plt.suptitle("Training Samples", fontsize=14)
    plt.tight_layout()
    plt.show()

## 4. Train YOLOv5n

Transfer learning from COCO-pretrained weights.

Expected training time: ~2.5 hours on T4 GPU for 300 epochs.

**Disconnect resilience:** Checkpoints are synced to Google Drive every 5 minutes via a background thread. If the Colab runtime is recycled, simply **re-run all cells from the top** -- the training cell will automatically detect the Drive backup and resume from the last saved epoch.

**Resume priority:** local `last.pt` > Google Drive `last.pt` > fresh start.

**Compatibility fixes applied automatically:**
- PyTorch >= 2.6: `torch.load()` patched to `weights_only=False`
- Pillow 10+: `font.getsize()` patched to `font.getbbox()`

In [ ]:
%cd /content/yolov5

import os, shutil, threading, tempfile, subprocess

# ============================================================
# Paths
# ============================================================
RUN_DIR = f"/content/yolov5/runs/{PROJECT_NAME}/{RUN_NAME}"
WEIGHTS_DIR = f"{RUN_DIR}/weights"
LOCAL_LAST = f"{WEIGHTS_DIR}/last.pt"
LOCAL_BEST = f"{WEIGHTS_DIR}/best.pt"
DRIVE_LAST = f"{DRIVE_BACKUP_DIR}/last.pt"
DRIVE_BEST = f"{DRIVE_BACKUP_DIR}/best.pt"

# ============================================================
# [1/4] Checkpoint resume detection
# Priority: local last.pt > Drive backup > fresh start
# ============================================================
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
os.makedirs(WEIGHTS_DIR, exist_ok=True)

resume_from = None

if os.path.exists(LOCAL_LAST):
    size_mb = os.path.getsize(LOCAL_LAST) / 1024 / 1024
    print(f"[Resume] Found local checkpoint: {LOCAL_LAST} ({size_mb:.1f} MB)")
    resume_from = LOCAL_LAST
elif os.path.exists(DRIVE_LAST):
    size_mb = os.path.getsize(DRIVE_LAST) / 1024 / 1024
    print(f"[Resume] Local checkpoint lost. Restoring from Drive ({size_mb:.1f} MB)...")
    shutil.copy2(DRIVE_LAST, LOCAL_LAST)
    if os.path.exists(DRIVE_BEST):
        shutil.copy2(DRIVE_BEST, LOCAL_BEST)
    resume_from = LOCAL_LAST
    print(f"[Resume] Checkpoint restored to {LOCAL_LAST}")
else:
    print("[Resume] No checkpoint found. Starting fresh training.")

# ============================================================
# [2/4] Background Drive sync thread
# Copies last.pt + best.pt to Drive every BACKUP_INTERVAL_MIN.
# Uses temp file + os.rename for atomic writes -- avoids corrupted
# backups if shutil.copy2 races with torch.save on the same file.
# ============================================================
_stop_sync = threading.Event()

def _safe_copy(src, dst):
    """Copy via temp file + rename to avoid partial/corrupted copies."""
    dst_dir = os.path.dirname(dst)
    fd, tmp_path = tempfile.mkstemp(dir=dst_dir, suffix=".tmp")
    try:
        os.close(fd)
        shutil.copy2(src, tmp_path)
        os.rename(tmp_path, dst)
    except Exception:
        try:
            os.unlink(tmp_path)
        except OSError:
            pass
        raise

def _sync_to_drive():
    """Periodically backup checkpoints to Google Drive."""
    while not _stop_sync.wait(BACKUP_INTERVAL_MIN * 60):
        for src, dst in [(LOCAL_LAST, DRIVE_LAST), (LOCAL_BEST, DRIVE_BEST)]:
            if os.path.exists(src):
                try:
                    _safe_copy(src, dst)
                except Exception as e:
                    print(f"[Backup] Warning: {e}")

_sync_thread = threading.Thread(target=_sync_to_drive, daemon=True)
_sync_thread.start()
print(f"[Backup] Drive sync started: every {BACKUP_INTERVAL_MIN} min -> {DRIVE_BACKUP_DIR}")

# ============================================================
# [3/4] Compatibility patches (idempotent)
# Mirrors training/patch_yolov5_compat.py (inline for Colab portability)
# ============================================================

def _patch_torch_load(filepath):
    with open(filepath, 'r') as f:
        content = f.read()
    original = content
    result = []
    i = 0
    while i < len(content):
        idx = content.find('torch.load(', i)
        if idx == -1:
            result.append(content[i:])
            break
        result.append(content[i:idx])
        start = idx + len('torch.load(')
        depth = 1
        j = start
        while j < len(content) and depth > 0:
            if content[j] == '(': depth += 1
            elif content[j] == ')': depth -= 1
            j += 1
        args_str = content[start:j-1]
        if 'weights_only' not in args_str:
            result.append(f'torch.load({args_str}, weights_only=False)')
        else:
            result.append(f'torch.load({args_str})')
        i = j
    content = ''.join(result)
    if content != original:
        with open(filepath, 'w') as f:
            f.write(content)
        return True
    return False

files_out = subprocess.run(
    ["grep", "-rl", "torch.load", "."],
    capture_output=True, text=True
).stdout.strip().split("\n")
patched = 0
for f in files_out:
    if f.endswith('.py'):
        if _patch_torch_load(f):
            patched += 1
            print(f"  Patched: {f}")
print(f"[Patch] torch.load: {patched} files fixed")

plots_py = "utils/plots.py"
with open(plots_py, 'r') as f:
    full_text = f.read()

if 'getbbox' in full_text:
    print("[Patch] Pillow getsize: already applied")
elif 'getsize' not in full_text:
    print("[Patch] Pillow getsize: not needed")
else:
    lines = full_text.splitlines(keepends=True)
    new_lines = []
    for line in lines:
        if 'self.font.getsize(label)' in line and 'try' not in line:
            indent = line[:len(line) - len(line.lstrip())]
            inner = indent + '    '
            new_lines.append(f'{indent}try:\n')
            new_lines.append(f'{inner}w, h = self.font.getsize(label)\n')
            new_lines.append(f'{indent}except AttributeError:\n')
            new_lines.append(f'{inner}bbox = self.font.getbbox(label)\n')
            new_lines.append(f'{inner}w, h = bbox[2] - bbox[0], bbox[3] - bbox[1]\n')
        else:
            new_lines.append(line)
    with open(plots_py, 'w') as f:
        f.writelines(new_lines)
    print("[Patch] Pillow getsize: applied")

# ============================================================
# [4/4] Train (auto-resume or fresh start)
# ============================================================
print()
if resume_from:
    print(f">>> Resuming training from checkpoint")
    !python train.py --resume "{resume_from}"
else:
    print(f">>> Starting fresh training: {EPOCHS} epochs")
    !python train.py \
        --data "{DATASET_ROOT}/data.yaml" \
        --cfg yolov5n.yaml \
        --weights yolov5n.pt \
        --img {IMG_SIZE} \
        --batch-size {BATCH_SIZE} \
        --epochs {EPOCHS} \
        --workers {WORKERS} \
        --project runs/{PROJECT_NAME} \
        --name {RUN_NAME} \
        --exist-ok \
        --cache ram

# ============================================================
# Final backup
# ============================================================
_stop_sync.set()
backed_up = []
for src, dst in [(LOCAL_LAST, DRIVE_LAST), (LOCAL_BEST, DRIVE_BEST)]:
    if os.path.exists(src):
        try:
            _safe_copy(src, dst)
            size_mb = os.path.getsize(dst) / 1024 / 1024
            backed_up.append(f"  {os.path.basename(dst)} ({size_mb:.1f} MB)")
        except Exception as e:
            print(f"[Backup] Final backup failed for {src}: {e}")

results_csv = f"{RUN_DIR}/results.csv"
if os.path.exists(results_csv):
    try:
        shutil.copy2(results_csv, f"{DRIVE_BACKUP_DIR}/results.csv")
        backed_up.append("  results.csv")
    except Exception:
        pass

if backed_up:
    print(f"\n[Backup] Final sync to Drive complete:")
    for item in backed_up:
        print(item)

## 5. Evaluate Results

In [ ]:
from IPython.display import Image, display

results_dir = f"/content/yolov5/runs/{PROJECT_NAME}/{RUN_NAME}"

for img_name, title in [
    ("results.png", "Training Curves"),
    ("confusion_matrix.png", "Confusion Matrix"),
    ("PR_curve.png", "PR Curve"),
    ("F1_curve.png", "F1 Curve"),
]:
    img_path = f"{results_dir}/{img_name}"
    if os.path.exists(img_path):
        print(f"\n{title}:")
        display(Image(filename=img_path, width=700))

In [ ]:
# Validation
!python val.py \
    --data "{DATASET_ROOT}/data.yaml" \
    --weights runs/{PROJECT_NAME}/{RUN_NAME}/weights/best.pt \
    --img 640 \
    --task val \
    --verbose

In [ ]:
# Visual detection results on val set
!python detect.py \
    --weights runs/{PROJECT_NAME}/{RUN_NAME}/weights/best.pt \
    --img 640 \
    --conf 0.25 \
    --source "{DATASET_ROOT}/val/images" \
    --project runs/{PROJECT_NAME} \
    --name val_detect \
    --exist-ok \
    --save-txt \
    --max-det 20

detect_dir = f"/content/yolov5/runs/{PROJECT_NAME}/val_detect"
if os.path.exists(detect_dir):
    det_imgs = [f for f in os.listdir(detect_dir)
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))][:8]
    if det_imgs:
        fig, axes = plt.subplots(2, 4, figsize=(20, 10))
        for ax, img_file in zip(axes.flat, det_imgs):
            img = cv2.imread(os.path.join(detect_dir, img_file))
            if img is not None:
                ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            ax.axis("off")
        plt.suptitle("Detection Results (Val Set)", fontsize=14)
        plt.tight_layout()
        plt.show()

## 6. Export ONNX for RKNN

Export with `--rknpu` flag (required for airockchip fork).

In [ ]:
%cd /content/yolov5

WEIGHTS = f"runs/{PROJECT_NAME}/{RUN_NAME}/weights/best.pt"

!python export.py \
    --weights {WEIGHTS} \
    --img-size 640 640 \
    --batch-size 1 \
    --rknpu \
    --include onnx

onnx_path = WEIGHTS.replace(".pt", ".onnx")
if os.path.exists(onnx_path):
    size_mb = os.path.getsize(onnx_path) / 1024 / 1024
    print(f"\nONNX exported: {onnx_path} ({size_mb:.2f} MB)")
else:
    print("ONNX export failed! If 'No module named onnxscript', run: !pip install onnxscript")

## 7. Convert to RKNN (INT8 Quantization)

Uses isolated virtualenv to avoid dependency conflicts with Colab.

In [ ]:
%%writefile /content/convert_rknn.py
import glob, os
from rknn.api import RKNN

# These are set via environment variables from the calling cell
ONNX_PATH = os.environ["ONNX_PATH"]
RKNN_PATH = os.environ["RKNN_PATH"]
DATASET_ROOT = os.environ["DATASET_ROOT"]

cal_images = sorted(glob.glob(f"{DATASET_ROOT}/train/images/*.jpg"))[:50]
cal_file = "/content/dataset.txt"
with open(cal_file, "w") as f:
    f.write("\n".join(cal_images))
print(f"Calibration images: {len(cal_images)}")

rknn = RKNN(verbose=False)
rknn.config(
    mean_values=[[0, 0, 0]],
    std_values=[[255, 255, 255]],
    target_platform="rv1106",
)

print("Loading ONNX...")
ret = rknn.load_onnx(model=ONNX_PATH)
assert ret == 0, f"Load ONNX failed: {ret}"

print("Building RKNN (INT8 quantization)...")
ret = rknn.build(do_quantization=True, dataset=cal_file)
assert ret == 0, f"Build failed: {ret}"

print("Exporting...")
ret = rknn.export_rknn(RKNN_PATH)
assert ret == 0, f"Export failed: {ret}"

rknn.release()
size_mb = os.path.getsize(RKNN_PATH) / 1024 / 1024
print(f"\nDone! {RKNN_PATH}: {size_mb:.2f} MB")

In [ ]:
import os

RKNN_NAME = "speed_signs_rv1106.rknn"
os.environ["ONNX_PATH"] = f"/content/yolov5/runs/{PROJECT_NAME}/{RUN_NAME}/weights/best.onnx"
os.environ["RKNN_PATH"] = f"/content/{RKNN_NAME}"
os.environ["DATASET_ROOT"] = DATASET_ROOT

!pip install virtualenv -q
!virtualenv /content/rknn_venv 2>/dev/null || (rm -rf /content/rknn_venv && virtualenv /content/rknn_venv)
!/content/rknn_venv/bin/pip install "setuptools<70" -q
!/content/rknn_venv/bin/pip install "onnx==1.16.2" -q
!/content/rknn_venv/bin/pip install rknn-toolkit2 -q

!/content/rknn_venv/bin/python3 /content/convert_rknn.py

## 8. Download Trained Model

In [ ]:
from google.colab import files
import shutil

# Copy best weights with clear names
weights_dir = f"/content/yolov5/runs/{PROJECT_NAME}/{RUN_NAME}/weights"
pt_src = f"{weights_dir}/best.pt"
onnx_src = f"{weights_dir}/best.onnx"
pt_dst = f"{weights_dir}/speed_signs.pt"
onnx_dst = f"{weights_dir}/speed_signs.onnx"

if os.path.exists(pt_src):
    shutil.copy2(pt_src, pt_dst)
if os.path.exists(onnx_src):
    shutil.copy2(onnx_src, onnx_dst)

print(f"Downloading speed_signs model artifacts...")
print("=" * 50)

for path, desc in [
    (pt_dst, "PyTorch weights"),
    (onnx_dst, "ONNX model"),
    (f"/content/{RKNN_NAME}", "RKNN model (INT8, RV1106)"),
]:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1024 / 1024
        print(f"  {os.path.basename(path)}: {size_mb:.2f} MB ({desc})")
        files.download(path)

print(f"\n{'='*50}")
print(f"  Deployment:")
print(f"{'='*50}")
print(f"  1. adb push {RKNN_NAME} /root/model/")
print(f"  2. adb push build/ai-hud /root/ai-hud")